# 💳 Finance Intelligence Pipeline — Interactive EDA

This notebook is a **companion to the automated EDA engine** (`src/eda/analysis.py`).
Use it for ad-hoc exploration, experimenting with new chart ideas, or debugging.

The pipeline must have been run at least once (`python pipeline.py --pdf <file.pdf>`) before this notebook will have data to show.

---

In [ ]:
import sys
sys.path.append('..')  # Allow imports from project root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from sqlalchemy import create_engine, text
from rich import print as rprint

# Project imports
from config.settings import settings
from config.merchants import CATEGORY_COLORS, CATEGORY_LABELS

# Notebook display settings
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.facecolor'] = '#0F1117'
plt.rcParams['axes.facecolor'] = '#1A1D27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#C8CDD6'
plt.rcParams['xtick.color'] = '#8B92A5'
plt.rcParams['ytick.color'] = '#8B92A5'
plt.rcParams['grid.color'] = '#2D3142'

print('✓ Environment ready')

## 1. Connect to Database

In [ ]:
engine = create_engine(settings.db.url, echo=False)

with engine.connect() as conn:
    # Main transactions table
    df = pd.read_sql(
        text('SELECT * FROM fact_transactions ORDER BY transaction_date'),
        conn,
        parse_dates=['transaction_date']
    )
    # Monthly cashflow view
    monthly = pd.read_sql(
        text('SELECT * FROM v_monthly_cashflow ORDER BY year, month'),
        conn
    )
    # Category totals
    categories = pd.read_sql(
        text('SELECT * FROM v_category_totals ORDER BY total_spent DESC'),
        conn
    )
    # ML predictions
    predictions = pd.read_sql(
        text('SELECT * FROM agg_monthly_spending WHERE predicted_spend IS NOT NULL ORDER BY year, month'),
        conn
    )

print(f'✓ Loaded {len(df):,} transactions')
print(f'  Date range: {df.transaction_date.min().date()} → {df.transaction_date.max().date()}')
print(f'  Categories: {df.category_code.nunique()}')
print(f'  Months: {len(monthly)}')

## 2. Quick Stats

In [ ]:
expenses = df[df['is_expense'] == 1]
income   = df[df['is_expense'] == 0]

print(f"Total Expenses:    {expenses['abs_amount'].sum():>12,.2f} PLN")
print(f"Total Income:      {income['abs_amount'].sum():>12,.2f} PLN")
print(f"Net Savings:       {(income['abs_amount'].sum() - expenses['abs_amount'].sum()):>12,.2f} PLN")
print(f"Avg. Transaction:  {expenses['abs_amount'].mean():>12,.2f} PLN")
print(f"Biggest Expense:   {expenses['abs_amount'].max():>12,.2f} PLN")
print(f"Current Balance:   {df.sort_values('transaction_date').iloc[-1]['balance_pln']:>12,.2f} PLN")

## 3. Monthly Income vs Expenses

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), facecolor='#0F1117')
ax.set_facecolor('#1A1D27')

x = range(len(monthly))
width = 0.35
ax.bar([i - width/2 for i in x], monthly['total_income'],   width, color='#06D6A0', alpha=0.85, label='Income')
ax.bar([i + width/2 for i in x], monthly['total_expenses'], width, color='#EF476F', alpha=0.85, label='Expenses')

ax2 = ax.twinx()
ax2.plot(x, monthly['net_savings'], color='#4CC9F0', linewidth=2.5, marker='o', markersize=5, label='Net Savings')
ax2.axhline(0, color='#FFD166', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_ylabel('Net Savings (PLN)', color='#4CC9F0')

ax.set_xticks(x)
ax.set_xticklabels(monthly['month_label'], rotation=45, ha='right')
ax.set_ylabel('PLN')
ax.set_title('Monthly Cash Flow — Income vs Expenses', fontsize=14, color='white', pad=12)
ax.legend(loc='upper left'); ax2.legend(loc='upper right')
ax.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

## 4. Spending by Category

In [ ]:
top_cats = categories.head(10)
colors = [CATEGORY_COLORS.get(c, '#7F8C8D') for c in top_cats['category_code']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), facecolor='#0F1117')
for ax in [ax1, ax2]:
    ax.set_facecolor('#1A1D27')

# Donut
wedges, _, autotexts = ax1.pie(
    top_cats['total_spent'], colors=colors,
    autopct='%1.1f%%', pctdistance=0.75,
    startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='#0F1117', linewidth=2)
)
for at in autotexts: at.set_color('white'); at.set_fontsize(9)
ax1.set_title('Category Breakdown', fontsize=13, color='white', pad=10)
ax1.legend(wedges, [CATEGORY_LABELS.get(c, c) for c in top_cats['category_code']],
           loc='center left', bbox_to_anchor=(1.05, 0.5), fontsize=9)

# Bar
ax2.barh(top_cats['category_code'][::-1], top_cats['total_spent'][::-1], color=colors[::-1], alpha=0.85)
ax2.set_title('Total Spend per Category (PLN)', fontsize=13, color='white', pad=10)
ax2.set_xlabel('PLN')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Day-of-Week Spending Patterns

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_avg = expenses.groupby('day_name')['abs_amount'].agg(['mean', 'sum', 'count']).reindex(day_order)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#0F1117')
titles = ['Avg Transaction (PLN)', 'Total Spend (PLN)', 'Transaction Count']
cols   = ['mean', 'sum', 'count']

for ax, col, title in zip(axes, cols, titles):
    ax.set_facecolor('#1A1D27')
    colors = ['#EF476F' if d in ['Saturday','Sunday'] else '#4CC9F0' for d in day_order]
    ax.bar(day_order, day_avg[col], color=colors, alpha=0.85)
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xticklabels(day_order, rotation=45, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Spending Patterns by Day of Week', fontsize=14, color='white', y=1.02)
plt.tight_layout()
plt.show()

## 6. ML Predictions vs Actual

In [ ]:
if not predictions.empty:
    pred_total = predictions.groupby(['year','month'])['predicted_spend'].sum().reset_index()
    pred_total['label'] = pred_total.apply(lambda r: f"{int(r['year'])}-{int(r['month']):02d}", axis=1)
    
    # Actual monthly totals
    actual = monthly[['year','month','total_expenses']].copy()
    actual['label'] = actual.apply(lambda r: f"{int(r['year'])}-{int(r['month']):02d}", axis=1)
    
    merged = actual.merge(pred_total[['label','predicted_spend']], on='label', how='outer').sort_values('label')
    
    fig, ax = plt.subplots(figsize=(14, 6), facecolor='#0F1117')
    ax.set_facecolor('#1A1D27')
    
    x = range(len(merged))
    ax.plot(x, merged['total_expenses'],   color='#EF476F', linewidth=2.5, marker='o', label='Actual')
    ax.plot(x, merged['predicted_spend'],  color='#4CC9F0', linewidth=2.5, marker='s', linestyle='--', label='Predicted')
    
    ax.set_xticks(x)
    ax.set_xticklabels(merged['label'], rotation=45, ha='right')
    ax.set_ylabel('Total Monthly Spend (PLN)')
    ax.set_title('Actual vs Predicted Monthly Spending', fontsize=14, color='white', pad=12)
    ax.legend(fontsize=11)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print('No predictions yet — run: python pipeline.py --pdf <file> --train-ml')

## 7. Transaction Deep Dive — Filter by Category

In [ ]:
# Change CATEGORY below to explore any category
CATEGORY = 'GROCERIES'  # Options: GROCERIES, DINING, TRANSPORT, BEAUTY, CLOTHING, etc.

cat_df = df[(df['category_code'] == CATEGORY) & (df['is_expense'] == 1)].copy()

print(f"Category: {CATEGORY}")
print(f"Total transactions: {len(cat_df)}")
print(f"Total spent:        {cat_df['abs_amount'].sum():,.2f} PLN")
print(f"Average:            {cat_df['abs_amount'].mean():,.2f} PLN")
print(f"Max single spend:   {cat_df['abs_amount'].max():,.2f} PLN")
print()

if 'merchant_name' in cat_df.columns:
    top_merchants = cat_df.groupby('merchant_name')['abs_amount'].sum().sort_values(ascending=False).head(10)
    print('Top 10 merchants:')
    for merchant, amount in top_merchants.items():
        print(f'  {merchant:<40} {amount:>10,.2f} PLN')

## 8. Export Raw Data to CSV

In [ ]:
# Export full transactions to CSV for further analysis
output_path = '../reports/transactions_export.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')  # utf-8-sig for Excel compatibility
print(f'✓ Exported {len(df):,} transactions to {output_path}')